In [ ]:
import pandas as pd
import numpy as np

import re
import time
import os
import pickle

import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df = pd.read_csv("WELFake_Cleaned.csv")

In [ ]:
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:")
print(df.duplicated().sum())

print("\nLabel distribution:")
print(df["label"].value_counts())
df.shape

['title', 'text', 'label']

Missing values:
title    524
text     582
label      0
dtype: int64

Duplicates:
9

Label distribution:
label
0    34791
1    28846
Name: count, dtype: int64


(63637, 3)

In [ ]:
print("Missing title:", df["title"].isna().sum())
print("Missing text:", df["text"].isna().sum())

df[df["text"].isna()].head(10)

Missing title: 524
Missing text: 582


,title,text,label
437,MARCO RUBIO Weighs In On Castro’s Death: “He d...,NaN,1
459,THIS ONE CREEPY TWEET FROM HILLARY Should Have...,NaN,1
924,DIRTY JOBS’ MIKE ROWE: Great Opportunities Out...,NaN,1
1010,MARK ZUCKERBERG Rides Shotgun with Dale Earnha...,NaN,1
1132,THE VIEW WOMEN Go Off The Rails: Trump ‘has to...,NaN,1
1321,PRICELESS! MILO DESTROYS Heckling Muslim Woman...,NaN,1
1574,WATTERS’ WORLD: “Do you have Obamacare?”…”How ...,NaN,1
1661,NaN,NaN,1
1828,MELANIA TRUMP GIVES POWERFUL SPEECH to Honor ‘...,NaN,1
1836,SCROOGE PASTOR HECKLES Kids Waiting In Line Fo...,NaN,1


In [ ]:
df[df["title"].isna()].head(10)

,title,text,label
68,NaN,"Marcus A Degenhart , mother Margaret ann Roth ...",1
156,NaN,I was okay with this guy until recently. Now h...,1
379,NaN,I wonder if I t will really happen?? if so the...,1
427,NaN,Obama and Hillary are all about deceit and lie...,1
841,NaN,"Except, of course, other US weaponry. Where di...",1
1221,NaN,I'd rather see him live and keep doing his thi...,1
1475,NaN,fu – ck the Yankee K I_I NT5 – ha ha ha,1
1502,NaN,Those Abrams tanks have been cooking off like ...,1
1590,NaN,Well we know that they think we are a basket f...,1
1611,NaN,More cover ups and lies! I do not trust any of...,1


In [ ]:
missing_title = df["title"].isna()
missing_text = df["text"].isna()

print("Both missing:", (missing_title & missing_text).sum())
print("Only title missing:", (missing_title & ~missing_text).sum())
print("Only text missing:", (~missing_title & missing_text).sum())

Both missing: 5
Only title missing: 519
Only text missing: 577


In [ ]:
df["title"] = df["title"].fillna("")

In [ ]:
print(
    "Empty titles:",
    (df["title"].fillna("").str.strip() == "").sum()
)

print(
    "Empty texts:",
    (df["text"].fillna("").str.strip() == "").sum()
)

Empty titles: 524
Empty texts: 582


In [ ]:
print("Missing title:", df["title"].isna().sum())
print("Missing text:", df["text"].isna().sum())

print("\nBoth missing:",
      (df["title"].isna() & df["text"].isna()).sum())

print("Only title missing:",
      (df["title"].isna() & ~df["text"].isna()).sum())

print("Only text missing:",
      (~df["title"].isna() & df["text"].isna()).sum())

Missing title: 0
Missing text: 582

Both missing: 0
Only title missing: 0
Only text missing: 582


In [ ]:
print("Empty titles:",
      (df["title"].fillna("").str.strip() == "").sum())

print("Empty texts:",
      (df["text"].fillna("").str.strip() == "").sum())

Empty titles: 524
Empty texts: 582


In [ ]:
print("\nExamples with missing text:")
display(df[df["text"].isna()][["title", "text", "label"]].head(10))


Examples with missing text:


,title,text,label
437,MARCO RUBIO Weighs In On Castro’s Death: “He d...,NaN,1
459,THIS ONE CREEPY TWEET FROM HILLARY Should Have...,NaN,1
924,DIRTY JOBS’ MIKE ROWE: Great Opportunities Out...,NaN,1
1010,MARK ZUCKERBERG Rides Shotgun with Dale Earnha...,NaN,1
1132,THE VIEW WOMEN Go Off The Rails: Trump ‘has to...,NaN,1
1321,PRICELESS! MILO DESTROYS Heckling Muslim Woman...,NaN,1
1574,WATTERS’ WORLD: “Do you have Obamacare?”…”How ...,NaN,1
1661,,NaN,1
1828,MELANIA TRUMP GIVES POWERFUL SPEECH to Honor ‘...,NaN,1
1836,SCROOGE PASTOR HECKLES Kids Waiting In Line Fo...,NaN,1


In [ ]:
print("\nExamples with missing title:")
display(df[df["title"].isna()][["title", "text", "label"]].head(10))


Examples with missing title:


,title,text,label


In [ ]:
missing_text_df = df[df["text"].isna()]

print("Missing-text records:", len(missing_text_df))

print("\nLabel distribution:")
print(missing_text_df["label"].value_counts())

print("\nLabel percentages:")
print(
    missing_text_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Missing-text records: 582

Label distribution:
label
1    581
0      1
Name: count, dtype: int64

Label percentages:
label
1    99.83
0     0.17
Name: proportion, dtype: float64


In [ ]:
print(
    missing_text_df["title"]
    .fillna("")
    .str.strip()
    .replace("", np.nan)
    .notna()
    .sum()
)

577


In [ ]:
display(
    missing_text_df[
        ["title", "label"]
    ].head(30)
)

,title,label
437,MARCO RUBIO Weighs In On Castro’s Death: “He d...,1
459,THIS ONE CREEPY TWEET FROM HILLARY Should Have...,1
924,DIRTY JOBS’ MIKE ROWE: Great Opportunities Out...,1
1010,MARK ZUCKERBERG Rides Shotgun with Dale Earnha...,1
1132,THE VIEW WOMEN Go Off The Rails: Trump ‘has to...,1
1321,PRICELESS! MILO DESTROYS Heckling Muslim Woman...,1
1574,WATTERS’ WORLD: “Do you have Obamacare?”…”How ...,1
1661,,1
1828,MELANIA TRUMP GIVES POWERFUL SPEECH to Honor ‘...,1
1836,SCROOGE PASTOR HECKLES Kids Waiting In Line Fo...,1


In [ ]:
empty_title_df = df[
    df["title"].fillna("").str.strip() == ""
]

print("Empty-title records:", len(empty_title_df))

print("\nLabel distribution:")
print(empty_title_df["label"].value_counts())

print("\nPercentages:")
print(
    empty_title_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Empty-title records: 524

Label distribution:
label
1    524
Name: count, dtype: int64

Percentages:
label
1    100.0
Name: proportion, dtype: float64


In [ ]:
df = df.dropna(subset=["text"]).copy()

In [ ]:
df["title"] = df["title"].fillna("")

In [ ]:
df["title"] = df["title"].str.strip()

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nEmpty titles:",
      (df["title"].str.strip() == "").sum())

print("\nEmpty texts:",
      (df["text"].str.strip() == "").sum())

Shape: (63055, 3)

Missing values:
title    0
text     0
label    0
dtype: int64

Empty titles: 519

Empty texts: 0


In [ ]:
df["combined_text"] = (
    df["title"] + " " + df["text"]
).str.strip()

In [ ]:
print("=" * 50)
print("FINAL DATASET CHECK")
print("=" * 50)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(
    df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nEmpty titles:")
print(
    (df["title"].str.strip() == "").sum()
)

print("\nEmpty texts:")
print(
    (df["text"].str.strip() == "").sum()
)

FINAL DATASET CHECK

Shape:
(63055, 4)

Columns:
['title', 'text', 'label', 'combined_text']

Missing values:
title            0
text             0
label            0
combined_text    0
dtype: int64

Duplicate rows:
5

Label distribution:
label
0    34790
1    28265
Name: count, dtype: int64

Label percentages:
label
0    55.17
1    44.83
Name: proportion, dtype: float64

Empty titles:
519

Empty texts:
0


In [ ]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 63055
Rows after: 63050
Duplicates removed: 5


In [ ]:
print("=" * 50)
print("FINAL DATASET")
print("=" * 50)

print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(
    df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nEmpty titles:",
      (df["title"].str.strip() == "").sum())

print("\nEmpty texts:",
      (df["text"].str.strip() == "").sum())

FINAL DATASET
Shape: (63050, 4)

Missing values:
title            0
text             0
label            0
combined_text    0
dtype: int64

Duplicate rows: 0

Label distribution:
label
0    34788
1    28262
Name: count, dtype: int64

Label percentages:
label
0    55.18
1    44.82
Name: proportion, dtype: float64

Empty titles: 519

Empty texts: 0


In [ ]:
import csv

df.to_csv(
    "WELFake_Cleanedfinal.csv"
)

In [ ]:
df2 = pd.read_csv(
    "WELFake_Cleanedfinal.csv"
)

In [ ]:
print("=" * 50)
print("DATASET VERIFICATION")
print("=" * 50)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(
    df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DATASET VERIFICATION

Shape:
(63050, 4)

Columns:
['title', 'text', 'label', 'combined_text']

Missing values:
title            0
text             0
label            0
combined_text    0
dtype: int64

Duplicate rows:
0

Label distribution:
label
0    34788
1    28262
Name: count, dtype: int64

Label percentages:
label
0    55.18
1    44.82
Name: proportion, dtype: float64
